In [12]:
from pathlib import Path
import os
from dotenv import load_dotenv

# 从当前目录向上查找项目根目录的 .env
for directory in [Path.cwd(), *Path.cwd().parents]:
    env_file = directory / ".env"
    if env_file.is_file():
        load_dotenv(env_file)
        print(f"已加载配置文件: {env_file}")
        break
else:
    raise FileNotFoundError("未找到项目根目录的 .env 文件")

已加载配置文件: d:\HuaweiMoveData\Users\鄢文浩\Desktop\HelloAgent\.env


In [3]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action的格式必须是以下之一：
1. 调用工具：function_name(arg_name="arg_value")
2. 结束任务：Finish[最终答案]

# 重要提示:
- 每次只输出一对Thought-Action
- Action必须在同一行，不要换行
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[最终答案] 格式结束

请开始吧！
"""


In [4]:
import requests

def get_weather(city: str) -> str:
    """
    通过调用 wttr.in API 查询真实的天气信息。
    """
    # API端点，我们请求JSON格式的数据
    url = f"https://wttr.in/{city}?format=j1"
    
    try:
        # 发起网络请求
        response = requests.get(url)
        # 检查响应状态码是否为200 (成功)
        response.raise_for_status() 
        # 解析返回的JSON数据
        data = response.json()
        
        # 提取当前天气状况
        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']
        
        # 格式化成自然语言返回
        return f"{city}当前天气:{weather_desc}，气温{temp_c}摄氏度"
        
    except requests.exceptions.RequestException as e:
        # 处理网络错误
        return f"错误:查询天气时遇到网络问题 - {e}"
    except (KeyError, IndexError) as e:
        # 处理数据解析错误
        return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"


In [5]:
import os
from tavily import TavilyClient

def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)
    
    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    
    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]
        
        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"


In [6]:
# 将所有工具函数放入一个字典，方便后续调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}


In [7]:
from openai import OpenAI

class OpenAICompatibleClient:
    """
    一个用于调用任何兼容OpenAI接口的LLM服务的客户端。
    """
    def __init__(self, model: str, api_key: str, base_url: str):
        self.model = model
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def generate(self, prompt: str, system_prompt: str) -> str:
        """调用LLM API来生成回应。"""
        print("正在调用大语言模型...")
        try:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ]
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                stream=False
            )
            answer = response.choices[0].message.content
            print("大语言模型响应成功。")
            return answer
        except Exception as e:
            print(f"调用LLM API时发生错误: {e}")
            return "错误:调用语言模型服务时出错。"


In [13]:
import re

# --- 1. 从 .env 读取 LLM 和 Tavily 配置 ---
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")
MODEL_ID = os.getenv("MODEL_ID")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

missing_config = [
    name for name, value in {
        "API_KEY": API_KEY,
        "BASE_URL": BASE_URL,
        "MODEL_ID": MODEL_ID,
        "TAVILY_API_KEY": TAVILY_API_KEY,
    }.items()
    if not value
]
if missing_config:
    raise ValueError(f".env 缺少配置: {', '.join(missing_config)}")

llm = OpenAICompatibleClient(
    model=MODEL_ID,
    api_key=API_KEY,
    base_url=BASE_URL,
)

# --- 2. 初始化 ---
user_prompt = "你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。"
prompt_history = [f"用户请求: {user_prompt}"]

print(f"用户输入: {user_prompt}\n" + "="*40)

# --- 3. 运行主循环 ---
for i in range(5): # 设置最大循环次数
    print(f"--- 循环 {i+1} ---\n")
    
    # 3.1. 构建Prompt
    full_prompt = "\n".join(prompt_history)
    
    # 3.2. 调用LLM进行思考
    llm_output = llm.generate(full_prompt, system_prompt=AGENT_SYSTEM_PROMPT)
    # 模型可能会输出多余的Thought-Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != llm_output.strip():
            llm_output = truncated
            print("已截断多余的 Thought-Action 对")
    print(f"模型输出:\n{llm_output}\n")
    prompt_history.append(llm_output)
    
    # 3.3. 解析并执行行动
    action_match = re.search(r"Action: (.*)", llm_output, re.DOTALL)
    if not action_match:
        observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    action_str = action_match.group(1).strip()

    if action_str.startswith("Finish"):
        final_answer = re.match(r"Finish\[(.*)\]", action_str).group(1)
        print(f"任务完成，最终答案: {final_answer}")
        break
    
    tool_name = re.search(r"(\w+)\(", action_str).group(1)
    args_str = re.search(r"\((.*)\)", action_str).group(1)
    kwargs = dict(re.findall(r'(\w+)=\"([^\"]*)\"', args_str))

    if tool_name in available_tools:
        observation = available_tools[tool_name](**kwargs)
    else:
        observation = f"错误:未定义的工具 '{tool_name}'"

    # 3.4. 记录观察结果
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)

用户输入: 你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。
--- 循环 1 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 用户需要查询北京今天的天气，然后根据天气推荐景点。我需要先获取北京的天气信息，再进行推荐。
Action: get_weather(city="北京")

Observation: 北京当前天气:Smoky haze，气温28摄氏度
--- 循环 2 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 已获取北京天气为Smoky haze（雾霾）。接下来需要根据这个天气情况查找合适的旅游景点。
Action: get_attraction(city="北京", weather="Smoky haze")

Observation: In smoky haze weather, Forbidden City and Temple of Heaven are ideal for indoor visits. These sites offer rich cultural experiences without exposure to poor air quality.
--- 循环 3 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:


Observation: 错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。
--- 循环 4 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 我已经获得了北京今天的天气和对应的景点推荐，信息足够回答用户。
Action: Finish[北京今天天气为雾霾（Smoky haze），气温28摄氏度。根据此天气，推荐您参观故宫和天坛，这些室内景点能让您免受空气质量不佳的影响，同时享受丰富的文化体验。]

任务完成，最终答案: 北京今天天气为雾霾（Smoky haze），气温28摄氏度。根据此天气，推荐您参观故宫和天坛，这些室内景点能让您免受空气质量不佳的影响，同时享受丰富的文化体验。


## 1. 记忆功能

- 在循环外维护 `agent_state`，保存用户偏好、已经推荐过的景点和拒绝次数。
- 每轮收到用户消息后，先从消息中提取偏好，再把记忆摘要加入 Prompt。
- 只保存完成任务需要的信息，并允许用户修改或清除偏好。

## 2. 售罄后的备选方案

- 将景点查询结果设计为结构化信息，至少包含 `name`、`available` 和 `reason`。
- 如果工具结果包含“售罄/无票/不可预订”，不要直接结束循环，而是追加 Observation。
- Observation 明确告诉 Agent：当前方案不可用，调用备选搜索工具，并排除已经推荐过的景点。

## 3. 连续拒绝三次后的反思

- 每当用户明确拒绝推荐，`rejected_count += 1`，同时记录被拒绝的景点。
- 达到 3 次时生成一条 Reflection Observation，让 Agent 总结拒绝原因。
- 重置计数，并改变策略，例如从“历史文化”切换到“自然/低预算/室内”等方向，而不是重复推荐。

这样，Thought-Action-Observation 循环变成：

```text
读取用户消息 -> 更新记忆 -> Thought -> Action -> 执行工具
-> 判断是否售罄/被拒绝 -> Observation -> 必要时 Reflection -> 下一轮
```

In [15]:
# 这是一个最小状态层：后续可以替换为数据库或向量记忆。
agent_state = {
    "preferences": {},
    "recommended": [],
    "rejected": [],
    "rejected_count": 0,
    "strategy": "先根据用户明确偏好推荐，再考虑天气、预算和可用性。",
}


def update_memory(user_message: str) -> None:
    """从用户消息中保存少量可解释的偏好信息。"""
    preference_rules = {
        "历史文化": ["历史", "文化", "博物馆", "古迹"],
        "自然风景": ["自然", "山", "湖", "公园", "风景"],
        "低预算": ["便宜", "省钱", "低预算", "预算低", "预算不要太高", "不太贵"],
        "室内": ["室内", "下雨", "不想晒", "避雨"],
    }
    for preference, keywords in preference_rules.items():
        if any(keyword in user_message for keyword in keywords):
            agent_state["preferences"][preference] = True


def remember_recommendation(name: str) -> None:
    if name and name not in agent_state["recommended"]:
        agent_state["recommended"].append(name)


def record_user_feedback(user_message: str, recommendation: str = "") -> str:
    """处理拒绝反馈；连续三次后返回 Reflection Observation。"""
    rejection_words = ["不要", "不喜欢", "不想去", "不合适", "拒绝", "换一个"]
    if not any(word in user_message for word in rejection_words):
        return ""

    agent_state["rejected_count"] += 1
    if recommendation:
        agent_state["rejected"].append(recommendation)

    if agent_state["rejected_count"] < 3:
        return f"用户拒绝了当前推荐。已连续拒绝 {agent_state['rejected_count']} 次，请避免重复推荐。"

    agent_state["strategy"] = (
        "用户连续拒绝了 3 次推荐。请先反思可能原因，主动询问预算、距离、兴趣和出行方式，"
        "再换一个明显不同的方向，不要只更换景点名称。"
    )
    agent_state["rejected_count"] = 0
    return f"Reflection: 用户连续拒绝了 3 个推荐：{agent_state['rejected']}。请调整推荐策略。"


def build_memory_observation() -> str:
    return (
        f"Memory: 用户偏好={agent_state['preferences']}; "
        f"已推荐={agent_state['recommended']}; "
        f"当前策略={agent_state['strategy']}"
    )


# 示例：把用户的新消息放在每轮循环开始处执行。
example_message = "我喜欢历史文化景点，但预算不要太高。"
update_memory(example_message)
print(build_memory_observation())

Memory: 用户偏好={'历史文化': True, '低预算': True}; 已推荐=[]; 当前策略=先根据用户明确偏好推荐，再考虑天气、预算和可用性。


In [ ]:
def recommend_with_fallback(city: str, weather: str) -> str:
    """景点售罄时，自动执行一次排除式备选搜索。"""
    first_result = get_attraction(city, weather)
    sold_out_words = ["售罄", "无票", "没票", "不可预订", "预约失败"]

    if not any(word in first_result for word in sold_out_words):
        return first_result

    excluded = ", ".join(agent_state["recommended"][-3:]) or "当前景点"
    fallback_query = (
        f"{city} {weather} 旅游景点推荐，要求有票或可预约，"
        f"避开这些景点：{excluded}，并说明门票和预约方式"
    )
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "Observation: 原推荐可能售罄，但未配置 TAVILY_API_KEY，无法搜索备选。"

    try:
        response = TavilyClient(api_key=api_key).search(
            query=fallback_query,
            search_depth="basic",
            include_answer=True,
        )
        fallback_result = response.get("answer") or "未找到可用的备选景点。"
        return f"Observation: 原推荐可能售罄，已触发备选搜索。\n{fallback_result}"
    except Exception as error:
        return f"Observation: 备选搜索失败，请重新选择景点。错误：{error}"


# 接入 Action 执行分支时，将原来的调用替换为：
# observation = recommend_with_fallback(city, weather)

# 接入每轮循环时，可以追加：
# update_memory(user_message)
# prompt_history.append(build_memory_observation())
# feedback_observation = record_user_feedback(user_message, recommendation)
# if feedback_observation:
#     prompt_history.append(feedback_observation)